# 🚀 Mini-Project 5: End-to-End EDA

**Objective**: Perform a complete Exploratory Data Analysis workflow on the famous Titanic dataset. Formulate hypotheses, engineer features, and visualize the findings.

**Skills Tested**:
- Entire Data Science Foundations skillset
- Asking business/research questions
- Visual storytelling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Load and Inspect Data
We will load the Titanic dataset directly from seaborn.

In [ ]:
df = sns.load_dataset('titanic')
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
df.head()

## 2. Handle Missing Data
The 'deck' column is mostly missing. 'age' and 'embark_town' have some missing.

In [ ]:
# Drop the deck column (too many missing)
df = df.drop('deck', axis=1)

# Fill Age with Median (less sensitive to outliers)
df['age'] = df['age'].fillna(df['age'].median())

# Fill embark_town with Mode (most frequent)
mode_town = df['embark_town'].mode()[0]
df['embark_town'] = df['embark_town'].fillna(mode_town)
df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])

print("Missing after clean:\n", df.isnull().sum())

## 3. Feature Engineering
Let's create some new features that might be highly predictive of survival.

In [ ]:
# 1. Family Size (sibsp = siblings/spouses, parch = parents/children)
df['family_size'] = df['sibsp'] + df['parch'] + 1

# 2. Is_Alone (Binary flag)
df['is_alone'] = (df['family_size'] == 1).astype(int)

# 3. Fare_Category (Binning the continuous fare)
df['fare_category'] = pd.qcut(df['fare'], q=4, labels=['Low', 'Medium', 'High', 'Premium'])

df[['family_size', 'is_alone', 'fare_category']].head()

## 4. Visual Storytelling: Who Survived?
Let's explore the mantra: "Women and children first" and "Class matters".

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Survival by Sex
sns.barplot(data=df, x='sex', y='survived', ax=axes[0, 0], palette='pastel')
axes[0, 0].set_title('Survival Rate by Gender')
axes[0, 0].set_ylabel('Survival Probability')

# 2. Survival by Passenger Class
sns.barplot(data=df, x='pclass', y='survived', hue='sex', ax=axes[0, 1], palette='pastel')
axes[0, 1].set_title('Survival Rate by Class and Gender')

# 3. Age Distribution of Survivors vs Non-Survivors
sns.violinplot(data=df, x='survived', y='age', ax=axes[1, 0], palette='muted')
axes[1, 0].set_title('Age Distribution by Survival')
axes[1, 0].set_xticks([0, 1])
axes[1, 0].set_xticklabels(['Died', 'Survived'])

# 4. Survival by Family Size
sns.pointplot(data=df, x='family_size', y='survived', ax=axes[1, 1], color='blue')
axes[1, 1].set_title('Survival Rate by Family Size')
axes[1, 1].set_ylabel('Survival Probability')

plt.tight_layout()
plt.show()

## 5. Correlation Analysis
Which numerical features correlate most with survival?

In [ ]:
# Convert boolean to int for correlation
df['adult_male'] = df['adult_male'].astype(int)
df['alone'] = df['alone'].astype(int)

numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr[['survived']].sort_values(by='survived', ascending=False), 
            annot=True, 
            cmap='coolwarm', 
            vmin=-1, vmax=1)
plt.title("Correlation with Survival")
plt.show()

print("\nInsights:")
print("- 'adult_male' has a strong negative correlation (-0.55), meaning adult men were highly unlikely to survive.")
print("- 'fare' has a positive correlation (0.26), indicating those who paid more (first class) were more likely to survive.")